# 1 - Mise à jour de la base de données

## 0 - Importation des modules

In [1]:
# Rechargement automatique des modules
%load_ext autoreload
%autoreload 2

# Modules de base
import os
import json
import pandas as pd
import sys
import yaml

# Ajout du chemin
sys.path.append('..')

# Importation des modules ad hoc
from dashboard_template_database.builders.schema import SchemaBuilder
from dashboard_template_database.builders.tables import DuckdbTablesBuilder
from dashboard_template_database.storage.loader import Loader
from dashboard_template_database.operations.updater_v2 import DatabaseUpdaterV2
from dashboard_template_database.operations.deleter_v2 import DatabaseDeleterV2

# Chargement du fichier de configurations
with open("../config.yaml") as file:
    config = yaml.safe_load(file)

# Chargement du fichier de parmaètres
with open("../parameters/labels.json") as file:
    labels = json.load(file)


## 1 - Mise à jour de la base de données 

### Données de mise à jour de la base de données

In [2]:
# Création de nouvelles données à insérer/mettre à jour
update_data = pd.DataFrame({
    'indicator': ['Gross Domestic Product', 'Private Consumption', 'Gross Domestic Product'],
    'country': ['Germany', 'Germany', 'France'],
    'date': pd.to_datetime(['2023-01-01', '2023-01-01', '2023-01-01']),
    'value': [100.5, 75.2, 150.8],
    'kind': ['forecast', 'forecast', 'forecast'],
    'horizon': [4.0, 4.0, 4.0],
    'week': [1.0, 1.0, 1.0],
    'model': ['XGBStandard', 'RandomForest', 'LassoCV'],
    'training': ['training', 'training', 'training']
})

update_data.head()

,indicator,country,date,value,kind,horizon,week,model,training
0,Gross Domestic Product,Germany,2023-01-01,100.5,forecast,4.0,1.0,XGBStandard,training
1,Private Consumption,Germany,2023-01-01,75.2,forecast,4.0,1.0,RandomForest,training
2,Gross Domestic Product,France,2023-01-01,150.8,forecast,4.0,1.0,LassoCV,training


### Initialisation de la classe de mise à jour

In [3]:
# Initialisation de l'updater
updater = DatabaseUpdaterV2(
    path=os.path.join('../', config['OUTPUT_DATA']),
    categorical_threshold=config['THRESHOLD'],
    enable_validation=True
)

# Affichage du nombre de lignes avant la mise à jour
print(f"Nombre de lignes dans la fact table avant mise à jour:")
print(updater.conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0])


2026-01-22 20:24:35,792 - WARNING - Could not set autocommit mode: Catalog Error: unrecognized configuration parameter "autocommit"

Did you mean: "allow_community_extensions"


Nombre de lignes dans la fact table avant mise à jour:
679968


### Mise à jour des données

Faire un cas avec création de dimension tables
Faire un cas avec suppression de dimension tables

In [4]:
# Mise à jour de la base de données avec les nouvelles données
success = updater.update_database(
    update_df=update_data,
    check_duplicates_db=True,
    check_duplicates_update=True,
    keep='last',
    use_batch_processing=False,
    use_transaction=True
)

# Affichage
if success:
    print("✓ Mise à jour effectuée avec succès")
    print(f"\nNombre de lignes dans la fact table après mise à jour:")
    print(builder.conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0])
else:
    print("✗ Échec de la mise à jour")

2026-01-22 20:24:38,613 - INFO - Validating preconditions for operation: update
2026-01-22 20:24:38,615 - ERROR - Critical validation issues found for update operation:
2026-01-22 20:24:38,617 - ERROR -   - No merge keys provided for update operation
2026-01-22 20:24:38,619 - ERROR - Pre-update validation failed


✗ Échec de la mise à jour


In [ ]:
# Vérification des données ajoutées pour 2023
# Création de la requête
query = """
SELECT 
    f.date,
    c.label as country,
    i.label as indicator,
    k.label as kind,
    f.value,
    f.horizon
FROM fact_table f
LEFT JOIN dim_country c ON f.country = c.value
LEFT JOIN dim_indicator i ON f.indicator = i.value
LEFT JOIN dim_kind k ON f.kind = k.value
WHERE YEAR(f.date) = 2023
ORDER BY f.date, c.label, i.label
"""
# Exécution de la requête
result_df = builder.conn.execute(query).fetchdf()

# Affichage
print(f"Données ajoutées pour l'année 2023 ({len(result_df)} lignes):")

result_df.head()

: 

: 

## 2 - Suppression des données

### Analyse du jeu de données avant suppression

In [ ]:
# Comptage des lignes avant suppression
total_rows = builder.conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
print(f"Nombre total de lignes: {total_rows}")

# Comptage des lignes pour la France
france_query = """
SELECT COUNT(*) 
FROM fact_table f
LEFT JOIN dim_country c ON f.country = c.value
WHERE c.label = 'France'
"""
france_rows = builder.conn.execute(france_query).fetchone()[0]
print(f"Nombre de lignes pour la France: {france_rows}")

# Répartition par pays
country_distribution = builder.conn.execute("""
SELECT c.label as country, COUNT(*) as count
FROM fact_table f
LEFT JOIN dim_country c ON f.country = c.value
GROUP BY c.label
ORDER BY count DESC
""").fetchdf()

print("\nRépartition par pays:")
country_distribution.head()

: 

: 

### Suppression des données

Faire un cas où cela n'entraine pas la création d'une table de dimension
Faire un cas où cela entraine la création d'une table de dimension

In [ ]:
# Initialisation du deleter
deleter = DatabaseDeleterV2(
    connection=builder.conn,
    categorical_threshold=config['THRESHOLD'],
    enable_validation=True,
    auto_cleanup=True
)

# Analyse de l'impact de la suppression
# Récupération de l'ID de la France depuis dim_country
france_id_query = "SELECT value FROM dim_country WHERE label = 'France'"
france_id = builder.conn.execute(france_id_query).fetchone()[0]

# Construction du filtre pour la France
filters = [('country', '=', france_id)]

# Analyse de l'impact
impact_report = deleter.get_deletion_impact(filters=filters)

# Affichage
print("Rapport d'impact de la suppression:")
print(f"  - Lignes affectées: {impact_report.get('rows_affected', 0)}")
print(f"  - Colonnes affectées: {impact_report.get('columns_affected', [])}")
print(f"  - Tables de dimension affectées: {impact_report.get('dimension_tables_affected', [])}")
print(f"  - Index affectés: {impact_report.get('indexes_affected', [])}")

if impact_report.get('warnings'):
    print("\nAvertissements:")
    for warning in impact_report['warnings']:
        print(f"  - {warning}")

if impact_report.get('recommendations'):
    print("\nRecommandations:")
    for recommendation in impact_report['recommendations']:
        print(f"  - {recommendation}")

# Suppression des lignes relatives à la France
deleted_rows = deleter.delete_rows(
    filters=filters,
    use_transaction=True,
    perform_cleanup=True
)

# Affichage
if deleted_rows >= 0:
    print(f"✓ Suppression effectuée avec succès")
    print(f"  Nombre de lignes supprimées: {deleted_rows}")
else:
    print("✗ Échec de la suppression")

: 

: 

### Vérification après suppression

In [ ]:
# Vérification après suppression
total_rows_after = builder.conn.execute("SELECT COUNT(*) FROM fact_table").fetchone()[0]
print(f"Nombre total de lignes après suppression: {total_rows_after}")
print(f"Différence: {total_rows - total_rows_after} lignes supprimées\n")

# Vérification qu'il n'y a plus de lignes pour la France
france_rows_after = builder.conn.execute(france_query).fetchone()[0]
print(f"Nombre de lignes pour la France après suppression: {france_rows_after}")

# Nouvelle répartition par pays
country_distribution_after = builder.conn.execute("""
SELECT c.label as country, COUNT(*) as count
FROM fact_table f
LEFT JOIN dim_country c ON f.country = c.value
GROUP BY c.label
ORDER BY count DESC
""").fetchdf()

# Affichage
print("\nNouvelle répartition par pays:")
country_distribution_after.head()

: 

: 